In [ ]:
from agent.base_agent.base_agent import BaseAgent
from typing import Dict, Optional, List
import logging
import numpy as np

logger = logging.getLogger(__name__)

class BaseAgentAStock(BaseAgent):
    """
    A-share specific trading agent with position concentration limit modification
    适配A股市场的交易代理，新增单只股票最大持仓比例限制（作业核心修改）
    """
    def __init__(self, config: Dict):
        super().__init__(config)
        self.market = "cn"  # A股市场标识，固定为cn
        self.initial_cash = config["agent_config"].get("initial_cash", 100000.0)  # 初始资金10万元
        self.current_cash = self.initial_cash
        self.positions: Dict[str, int] = {}  # 持仓记录：股票代码→持股数量
        self.max_single_position_ratio = 0.2  # 作业核心修改：单只股票最大持仓比例（20%）
        logger.info(f"A-Share Agent initialized with initial cash: {self.initial_cash} CNY, max single position ratio: {self.max_single_position_ratio}")

    def calculate_portfolio_value(self) -> float:
        """计算当前总资产价值（现金+持仓股票市值）"""
        portfolio_value = self.current_cash
        for symbol, quantity in self.positions.items():
            if quantity <= 0:
                continue
            # 调用价格工具获取当前股票价格
            price_data = self.tool_check_price(symbol=symbol)
            if price_data and "current_price" in price_data:
                portfolio_value += quantity * price_data["current_price"]
        return portfolio_value

    def check_position_limit(self, symbol: str, buy_amount: int, buy_price: float) -> int:
        """
        作业核心修改：检查并调整买入数量，确保单只股票持仓不超过总资产的20%
        Args:
            symbol: 股票代码
            buy_amount: 拟买入数量
            buy_price: 买入价格
        Returns: 调整后的最大可买入数量
        """
        total_asset = self.calculate_portfolio_value()  # 当前总资产（现金+持仓）
        current_position_value = 0.0
        
        # 计算该股票当前持仓市值
        if symbol in self.positions and self.positions[symbol] > 0:
            current_position_value = self.positions[symbol] * buy_price
        
        # 计算拟买入后的总持仓市值
        proposed_position_value = current_position_value + (buy_amount * buy_price)
        max_allowed_value = total_asset * self.max_single_position_ratio  # 最大允许持仓市值
        
        # 如果拟买入后超过限制，调整最大可买入数量
        if proposed_position_value > max_allowed_value:
            excess_value = proposed_position_value - max_allowed_value
            excess_amount = int(np.ceil(excess_value / buy_price))
            adjusted_amount = buy_amount - excess_amount
            logger.warning(
                f"Position limit exceeded for {symbol}! "
                f"Proposed value: {proposed_position_value:.2f} CNY, "
                f"Max allowed: {max_allowed_value:.2f} CNY. "
                f"Adjusted buy amount from {buy_amount} to {adjusted_amount}"
            )
            return max(adjusted_amount, 0)  # 确保调整后数量非负
        return buy_amount

    def tool_trade(self, action: str, symbol: str, amount: int) -> Dict:
        """
        重写交易工具调用逻辑，集成持仓限制检查（作业核心修改）
        支持A股T+1规则、100股整数倍交易
        """
        # A股基础规则校验：100股整数倍交易
        if amount % 100 != 0:
            logger.error(f"A-Share trade error: Amount {amount} must be multiples of 100")
            return {"status": "failed", "reason": "Amount must be multiples of 100"}
        
        # 获取当前股票买入/卖出价格
        price_data = self.tool_check_price(symbol=symbol)
        if not price_data:
            logger.error(f"Failed to get price for {symbol}")
            return {"status": "failed", "reason": "Price data not available"}
        
        if action == "buy":
            buy_price = price_data.get("current_buy_price", price_data["current_price"])
            total_cost = amount * buy_price
            
            # 作业核心修改：检查持仓比例限制
            adjusted_amount = self.check_position_limit(symbol, amount, buy_price)
            if adjusted_amount <= 0:
                return {"status": "failed", "reason": "Position limit reached, no valid amount to buy"}
            
            # 校验现金是否充足
            if total_cost > self.current_cash:
                logger.warning(f"Insufficient cash: Need {total_cost:.2f} CNY, Available {self.current_cash:.2f} CNY")
                return {"status": "failed", "reason": "Insufficient cash"}
            
            # 执行买入
            self.current_cash -= adjusted_amount * buy_price
            self.positions[symbol] = self.positions.get(symbol, 0) + adjusted_amount
            logger.info(
                f"Buy success: {symbol} x {adjusted_amount} shares, "
                f"Price: {buy_price:.2f} CNY, "
                f"Total cost: {adjusted_amount * buy_price:.2f} CNY, "
                f"Remaining cash: {self.current_cash:.2f} CNY"
            )
            return {"status": "success", "action": "buy", "symbol": symbol, "amount": adjusted_amount, "price": buy_price}
        
        elif action == "sell":
            # A股T+1规则：当日买入不能当日卖出（简化校验）
            if symbol not in self.positions or self.positions[symbol] == 0:
                logger.error(f"No position to sell for {symbol}")
                return {"status": "failed", "reason": "No available position to sell"}
            
            # 校验卖出数量不超过持仓
            if amount > self.positions[symbol]:
                logger.warning(f"Sell amount {amount} exceeds position {self.positions[symbol]}")
                amount = self.positions[symbol]
            
            sell_price = price_data.get("current_sell_price", price_data["current_price"])
            total_revenue = amount * sell_price
            
            # 执行卖出
            self.current_cash += total_revenue
            self.positions[symbol] -= amount
            if self.positions[symbol] == 0:
                del self.positions[symbol]
            
            logger.info(
                f"Sell success: {symbol} x {amount} shares, "
                f"Price: {sell_price:.2f} CNY, "
                f"Total revenue: {total_revenue:.2f} CNY, "
                f"Current cash: {self.current_cash:.2f} CNY"
            )
            return {"status": "success", "action": "sell", "symbol": symbol, "amount": amount, "price": sell_price}
        
        else:
            return {"status": "failed", "reason": "Invalid action (only buy/sell supported)"}

    def run(self, date: str) -> Dict:
        """执行单日A股交易决策（继承BaseAgent逻辑，保持原流程）"""
        logger.info(f"Starting A-Share agent run for date: {date}")
        # 1. 获取市场观察（持仓+价格）
        observation = self.get_observation(date)
        # 2. 基于ReAct范式进行自主推理（调用Search/News工具）
        reasoning = self.autonomous_reasoning(observation)
        # 3. 执行交易动作
        action_result = self.execute_action(reasoning)
        # 4. 返回当日交易结果
        return {
            "date": date,
            "observation": observation,
            "reasoning": reasoning,
            "action_result": action_result,
            "current_positions": self.positions,
            "current_cash": self.current_cash,
            "total_asset": self.calculate_portfolio_value()
        }

# 作业新增：注册自定义代理（便于项目调用）
AGENT_REGISTRY = {
    "BaseAgentAStock": {
        "module": "agent.base_agent_astock.base_agent_astock",
        "class": "BaseAgentAStock"
    }
}